### Import Statements

In [1]:
import numpy as np
import pandas as pd

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import json
%matplotlib inline

### Set matplotlib text export settings for Adobe Illustrator

In [2]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

#### Pandas Viewing Settings

In [3]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [4]:
plt.style.use('../mgm.v1.mplstyle')

## Define paths to sample metadata files

In [5]:
!ls -1 ../../Data

220813_MtbEpitopes
231121_HybridMtbAsm_QCPass_Meta_Set3
231121.InputAsmTSVs.MtbSetV3.151CI
241024.MutSpectraAnalysis.MUSICAL
241030.Mtb151CI.AllVariants.Anno.V1
250910.PPE18.netMHCpanII.PredictionsForRecombinantSeqs
H37Rv.NonUniqueSeqRegions.V1
TBP22.22CI.GCEVerfIsolates.Metadata
Tgen1K_WGS_RunMetadata


In [6]:
Repo_DataDir = "../../Data"
InputAsmPath_Dir = f"{Repo_DataDir}/231121.InputAsmTSVs.MtbSetV3.151CI"

MtbSetV3_151CI_InputAsmPATHs_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAndSRAsm.FAPATHs.V1.tsv"

MtbSetV3_151CI_AsmSumm_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAsm.AsmSummary.V2.tsv"


## PARSE PATHs FOR ALL assemblies processed by this pipeline

In [7]:
WGA158CI_LRandSR_Asm_Path_DF = pd.read_csv(MtbSetV3_151CI_InputAsmPATHs_TSV, sep = "\t")

WGA158CI_LRandSR_Asm_Path_DF.columns = ['SampleID', 'Dataset_Tag',
                                        'Genome_LR_ASM_PATH', 'Genome_SR_ASM_PATH']


In [8]:
WGA158CI_LRandSR_Asm_Path_DF.head(1)

,SampleID,Dataset_Tag,Genome_LR_ASM_PATH,Genome_SR_ASM_PATH
0,N0072,ChinerOms_2019,/n/data1/hms/dbmi/farhat/mm774/Projects/231121...,/n/data1/hms/dbmi/farhat/mm774/Projects/231121...


## Parse sample Metadata (N = 151)

In [9]:

WGA151CI_AsmSummary_DF = pd.read_csv(MtbSetV3_151CI_AsmSumm_TSV, sep = "\t")

SampleIDs_151CI_SOI = list( WGA151CI_AsmSummary_DF["SampleID"].values )
WGA151CI_SampleIDs = SampleIDs_151CI_SOI

#print(','.join(SampleIDs_151CI_SOI) )

ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)
WGA151CI_AsmSummary_DF.shape

(151, 7)

In [10]:
WGA151CI_AsmSummary_DF.head(3)

,SampleID,numContigs_Complete,Flye_CircContig_Cov,PrimaryLineage,Lineage,Dataset_Tag,AsmApproach
0,N0072,1,358,lineage1,"lineage1,lineage1.1,lineage1.1.2",ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon
1,N0153,1,372,lineage1,"lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1",ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon
2,TB3113,1,933,lineage2,"lineage2,lineage2.2,lineage2.2.1",TB_Portals_24CI_R1,PBrs2_LR_Flye_I3_SR_Pilon


#### What is the lineage breakdown?

In [11]:
WGA151CI_AsmSummary_DF["PrimaryLineage"].value_counts()

PrimaryLineage
lineage4    62
lineage2    61
lineage1    15
lineage3     7
lineage6     3
lineage5     2
lineage8     1
Name: count, dtype: int64

#### What is the datasets are contributing to our 151 Mtb isolates?

In [12]:
WGA151CI_AsmSummary_DF["Dataset_Tag"].value_counts()

Dataset_Tag
Hall2022                78
TB_Portals_24CI_R1      21
Peker2021               17
Farhat_Peru_2019        13
ChinerOms_2019          12
TRUST_PB_Set1            8
Lee2020_Elife            1
Ngabonziza_Lin8_2020     1
Name: count, dtype: int64

#### What is the breakdown of hybrid assembly approaches used?
- 1 - `PBrs2_LR_Flye_I3_SR_Pilon` (N = 48) <br>
   PacBio RSII subreads assembled and polished (Flye), then polished w/ short-reads (Pilon)

- 2 - `PBccs_LR_Flye_I3_SR_Pilon` (N = 8) <br>
   PacBio Sequel II HiFi reads assembled and polished (Flye), then polished w/ short-reads (Pilon)
  
- 3 - `ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish` (N= 95) <br>
   Oxford Nanopore (9.4.1) reads assembled and polished (Flye, Medaka), then polished w/ short-reads (Pilon, PolyPolish)


In [13]:
WGA151CI_AsmSummary_DF["AsmApproach"].value_counts() 

AsmApproach
ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish    95
PBrs2_LR_Flye_I3_SR_Pilon                48
PBccs_LR_Flye_I3_SR_Pilon                 8
Name: count, dtype: int64

## Define PATHs to ALL assembly QC and info TSVs for the different sets of isolates

In [14]:
Repo_DataDir = "../../Data"

HybridMtbAsm_SetV3_MetaDir = Repo_DataDir + "/231121_HybridMtbAsm_QCPass_Meta_Set3"    

PBSR_48CI_AsmSumm_TSV_PATH = f"{HybridMtbAsm_SetV3_MetaDir}/231121.PBSR.QCPass.48CI.AsmSummary.V3.tsv"

Peker18CI_AsmSummary_TSV_PATH = HybridMtbAsm_SetV3_MetaDir + "/220720.Peker2021.17CI.AssemblySummary.Filt.V2.tsv"

Hall80CI_AsmSummary_TSV_PATH = HybridMtbAsm_SetV3_MetaDir + "/220720.Hall2022.78CI.AsmSummary.V2.Filt.tsv"

TRUST_PBSet1_8CI_AsmSumm_TSV_PATH = f"{HybridMtbAsm_SetV3_MetaDir}/220918.TRUST.PBSet1.QCPass.8CI.AsmSummary.V2.tsv"


In [15]:
!ls -1 $HybridMtbAsm_SetV3_MetaDir  

220720.Hall2022.78CI.AsmSummary.V2.Filt.tsv
220720.Hall2022.80CI.AsmSummary.V2.tsv
220720.Peker2021.17CI.AssemblySummary.Filt.V2.tsv
220720.Peker2021.18CI.AssemblySummary.V2.tsv
220918.TRUST.PBSet1.QCPass.8CI.AsmSummary.V2.tsv
231121.PBSR.QCPass.48CI.AsmSummary.V3.tsv


In [16]:
!ls -1 $Repo_DataDir/231121_HybridMtbAsm_QCPass_Meta_Set3_AsmPolishStats_OLD

ls: cannot access '../../Data/231121_HybridMtbAsm_QCPass_Meta_Set3_AsmPolishStats_OLD': No such file or directory


In [17]:
48 + 78 + 17 + 8

151

# 0) Read in main metadata sheet from Brendan

These sheets have a mapping between the `DNA-` IDs I have been using to the `TBP-` IDs we want to use


# 1.A) Parse Assembly INFO for PB-Mtb dataset (N = 48) 

In [18]:
PBSR_48CI_AsmSumm_TSV_PATH

'../../Data/231121_HybridMtbAsm_QCPass_Meta_Set3/231121.PBSR.QCPass.48CI.AsmSummary.V3.tsv'

In [19]:
PMP48CI_AsmSumm = pd.read_csv(PBSR_48CI_AsmSumm_TSV_PATH, sep = "\t") 

PMP48CI_AsmSumm = PMP48CI_AsmSumm.sort_values("PrimaryLineage_PB")

PMP48CI_AsmSumm["AsmApproach"] = "PBrs2_LR_Flye_I3_SR_Pilon"

PMP48CI_AsmSumm["Lineage"] = PMP48CI_AsmSumm["LineageCall_PacBio"]

PMP48CI_AsmSumm["PrimaryLineage"] = PMP48CI_AsmSumm["PrimaryLineage_PB"]

PMP48CI_AsmSumm["Dataset_Tag"] = PMP48CI_AsmSumm["Dataset_Tag"].replace("TB_Portals_24CI_R1", "TB_Portals_2020")

PMP48CI_SampleIDs = list( PMP48CI_AsmSumm["SampleID"].values )
PBRS_48CI_SampleIDs = list( PMP48CI_AsmSumm["SampleID"].values )

print("# of total samples:", len(PMP48CI_SampleIDs) )

print(','.join(PMP48CI_SampleIDs) )

# Make sample to lineage mapping dict
PMP48CI_ID_To_IlluminaAvrgCov_Dict = dict(PMP48CI_AsmSumm[['SampleID', 'IlluminaWGSToH37rv_AvrgCov']].values)                     
PMP48CI_ID_To_Lineage_Dict = dict(PMP48CI_AsmSumm[['SampleID', 'PrimaryLineage_PB']].values)
PMP48CI_ID_To_Dataset_Dict = dict(PMP48CI_AsmSumm[['SampleID', 'Dataset_Tag']].values)

PBRS48CI_SampleID_V1_to_V2_Dict = dict(PMP48CI_AsmSumm[['Alt_SampleID', 'SampleID']].values)
PBRS48CI_SampleID_V2_to_V1_Dict = dict(PMP48CI_AsmSumm[['SampleID', 'Alt_SampleID']].values)


PMP48CI_AsmSumm["Mean_SR_Cov"] = PMP48CI_AsmSumm["IlluminaWGSToH37rv_AvrgCov"]



# of total samples: 48
N0072,N0153,TB3113,TB1236,TB2659,TB2780,TB1612,TB2512,TB2981,TB3091,M0003941_3,TB3368,N0145,N0155,TB2995,TB3396,N0004,N1274,N0054,02_R1179,01_R1134,M0017522_5,M0016395_7,M0010874_7,02_R1708,02_R0894,01_R1430,M0014888_3,02_R1896,TB4620,TB3162,MT_0080,TB3054,TB3251,M0016737_0,TB2661,TB3237,TB3169,TB3386,TB3334,M0011368_9,TB2968,N1272,N1176,N1202,N1177,N0091,RW-TB008


In [20]:
!ls -1 ../../Data/231121_HybridMtbAsm_QCPass_Meta_Set3_AsmPolishStats_OLD

ls: cannot access '../../Data/231121_HybridMtbAsm_QCPass_Meta_Set3_AsmPolishStats_OLD': No such file or directory


In [21]:
PMP48CI_AsmSumm.head(4)

,Alt_SampleID,numContigs_Complete,circContig_Length,circContig_Cov,PacBio_Subread_Median_Length,LineageCall_Illumina,LineageCall_PacBio,F2_Illumina,F2_PacBio,ANI_I3,ANI_I3_PP,IlluminaWGSToH37rv_AvrgCov,PacBio_Subreads_H37Rv_AvrgCov,NumAnno_ORFs_PB_PilonPolished,NumAnno_ORFs_PB_DeNovo,GCcontent_PB_PP_GBK,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,PrimaryLineage_PB,PrimaryLineage_Ill,Dataset_Tag,AsmApproach,Lineage,PrimaryLineage,SampleID,Mean_SR_Cov
0,N0072,1,4421406,358,2413.0,"lineage1,lineage1.1,lineage1.1.2","lineage1,lineage1.1,lineage1.1.2",0.021909,0.263559,99.8849,99.8852,112,348,4047,4051,65.610969,11,0,4,3,1,7,7,lineage1,lineage1,ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon,"lineage1,lineage1.1,lineage1.1.2",lineage1,N0072,112
1,N0153,1,4389210,372,2027.0,"lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1","lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1",0.023551,0.272487,99.8687,99.8692,98,370,4037,4043,65.612081,53,19,2,1,1,32,32,lineage1,lineage1,ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon,"lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1",lineage1,N0153,98
15,DNA089,1,4420000,933,9945.0,"lineage2,lineage2.2,lineage2.2.1","lineage2,lineage2.2,lineage2.2.1",0.009493,0.200000,99.8898,99.8920,65,1573,4078,4107,65.609800,79,2,0,0,0,77,76,lineage2,lineage2,TB_Portals_2020,PBrs2_LR_Flye_I3_SR_Pilon,"lineage2,lineage2.2,lineage2.2.1",lineage2,TB3113,65
13,ZRB10578980,1,4413217,374,3790.0,"lineage2,lineage2.2,lineage2.2.1","lineage2,lineage2.2,lineage2.2.1",0.009687,0.216928,99.8957,99.8918,69,363,4066,4106,65.606149,106,4,16,10,2,86,85,lineage2,lineage2,TB_Portals_2020,PBrs2_LR_Flye_I3_SR_Pilon,"lineage2,lineage2.2,lineage2.2.1",lineage2,TB1236,69


In [22]:
PMP48CI_AsmSumm["Dataset_Tag"].value_counts()  

Dataset_Tag
TB_Portals_2020         21
Farhat_Peru_2019        13
ChinerOms_2019          12
Lee2020_Elife            1
Ngabonziza_Lin8_2020     1
Name: count, dtype: int64

In [23]:
PMP48CI_AsmSumm.sort_values("SampleID").head(4)

,Alt_SampleID,numContigs_Complete,circContig_Length,circContig_Cov,PacBio_Subread_Median_Length,LineageCall_Illumina,LineageCall_PacBio,F2_Illumina,F2_PacBio,ANI_I3,ANI_I3_PP,IlluminaWGSToH37rv_AvrgCov,PacBio_Subreads_H37Rv_AvrgCov,NumAnno_ORFs_PB_PilonPolished,NumAnno_ORFs_PB_DeNovo,GCcontent_PB_PP_GBK,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,PrimaryLineage_PB,PrimaryLineage_Ill,Dataset_Tag,AsmApproach,Lineage,PrimaryLineage,SampleID,Mean_SR_Cov
34,01_R1134,1,4405134,138,4826.0,lineage4,lineage4,0.009350,0.241268,99.9195,99.9360,62,130,4065,4550,65.615689,1034,2,1031,1022,6,1,1,lineage4,lineage4,Farhat_Peru_2019,PBrs2_LR_Flye_I3_SR_Pilon,lineage4,lineage4,01_R1134,62
40,01_R1430,1,4409128,227,6009.0,"lineage4,lineage4.4,lineage4.4.1","lineage4,lineage4.4,lineage4.4.1",0.005372,0.295069,99.9284,99.9290,253,212,4070,4119,65.613390,93,0,90,89,0,3,3,lineage4,lineage4,Farhat_Peru_2019,PBrs2_LR_Flye_I3_SR_Pilon,"lineage4,lineage4.4,lineage4.4.1",lineage4,01_R1430,253
39,02_R0894,1,4422314,191,4836.0,"lineage4,lineage4.1,lineage4.1.2,lineage4.1.2.1","lineage4,lineage4.1,lineage4.1.2,lineage4.1.2.1",0.013648,0.304220,99.9224,99.9284,141,178,4076,4180,65.635659,190,0,189,188,1,1,1,lineage4,lineage4,Farhat_Peru_2019,PBrs2_LR_Flye_I3_SR_Pilon,"lineage4,lineage4.1,lineage4.1.2,lineage4.1.2.1",lineage4,02_R0894,141
33,02_R1179,1,4393467,249,5543.0,"lineage4,lineage4.3,lineage4.3.3","lineage4,lineage4.3,lineage4.3.3",0.006001,0.306351,99.9295,99.9397,186,231,4052,4188,65.614352,309,0,307,305,1,2,2,lineage4,lineage4,Farhat_Peru_2019,PBrs2_LR_Flye_I3_SR_Pilon,"lineage4,lineage4.3,lineage4.3.3",lineage4,02_R1179,186


### Make a TGEN_Seq_ID to TBP_ID mapping Dict (For TB Portals Isolates)

In [24]:
TBP_TGEN_To_TBP_ID_Dict = dict(PMP48CI_AsmSumm.query("Dataset_Tag == 'TB_Portals_2020' ")[['Alt_SampleID', 'SampleID']].values)

In [25]:
TBP_TGEN_To_TBP_ID_Dict

{'DNA089': 'TB3113',
 'ZRB10578980': 'TB1236',
 'AZE_02_041': 'TB2659',
 'AZE_02_067': 'TB2780',
 'ARR1960': 'TB1612',
 'DNA096': 'TB2512',
 'DNA114': 'TB2981',
 'DNA019_Vash': 'TB3091',
 'DNA075': 'TB3368',
 'DNA028': 'TB2995',
 'DNA091': 'TB3396',
 'DNA054': 'TB4620',
 'DNA182': 'TB3162',
 'DNA124': 'TB3054',
 'DNA044': 'TB3251',
 'AZE_02_042': 'TB2661',
 'DNA020': 'TB3237',
 'DNA188': 'TB3169',
 'DNA086': 'TB3386',
 'DNA019_Rose': 'TB3334',
 'DNA120': 'TB2968'}

# 1.B) Parse Assembly INFO for Peker18CI dataset (N = 18)

In [26]:

Pek18CI_AsmSumm = pd.read_csv(Peker18CI_AsmSummary_TSV_PATH, sep = "\t")

Pek18CI_AsmSumm = Pek18CI_AsmSumm.sort_values("Lineage_ONTAsm_WiPilonPolish")

Pek18CI_SampleIDs = list( Pek18CI_AsmSumm["SampleID"].values )

Pek18CI_AsmSumm["AsmApproach"] = "ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish"

Pek18CI_AsmSumm["Lineage"] = Pek18CI_AsmSumm["Lineage_ONTAsm_WiPilonPolish"]

Pek18CI_AsmSumm["PrimaryLineage"] = Pek18CI_AsmSumm["PrimaryLineage_ONTAsm_WiPilonPolish"]

print("# of total samples:", len(Pek18CI_SampleIDs) )

print(', '.join(Pek18CI_SampleIDs) )

# Make sample to lineage mapping dict
Pek18CI_ID_To_PrimLineage_Dict = dict(Pek18CI_AsmSumm[['SampleID', 'PrimaryLineage_ONTAsm_WiPilonPolish']].values)
Pek18CI_ID_To_Lineage_Dict = dict(Pek18CI_AsmSumm[['SampleID', 'Lineage_ONTAsm_WiPilonPolish']].values)
Pek18CI_ID_To_Dataset_Dict = dict(Pek18CI_AsmSumm[['SampleID', 'Dataset_Tag']].values)


Pek18CI_AsmSumm["Mean_SR_Cov"] = Pek18CI_AsmSumm["IlluminaCov_To_ONTAsm"]


# of total samples: 17
9050-05, 4549-04, 696-05, 702-06, 706-05, 8129-04, 3003-06, 8651-04, QC-5, QC-9, QC-3, QC-8, QC-10, QC-4, QC-7, QC-1, QC-6


In [27]:
Pek18CI_AsmSumm.head(3)

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_ONTAsm,Lineage_ONTAsm_WiPilonPolish,FlyeI3_dnaA_Found,FlyeI3M_dnaA_Found,FlyeI3MPP_dnaA_Found,IlluminaCov_To_ONTAsm,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,NumChanges_PolyPolished,PrimaryLineage_ONTAsm_WiPilonPolish,Dataset_Tag,AsmApproach,Lineage,PrimaryLineage,Mean_SR_Cov
0,9050-05,1,4417291,376,380,4716,1361,lineage2.2.1,lineage2.2.1,False,False,True,130,8556,250,540,515,23,7766,7344,107,lineage2,Peker2021,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,lineage2.2.1,lineage2,130
14,4549-04,1,4423772,302,310,3036,620,lineage2.2.1,lineage2.2.1,False,False,True,83,11817,297,437,409,27,11083,10394,94,lineage2,Peker2021,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,lineage2.2.1,lineage2,83
13,696-05,1,4422244,299,306,3493,931,lineage2.2.1,lineage2.2.1,False,False,True,102,11609,313,453,431,22,10843,10185,109,lineage2,Peker2021,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,lineage2.2.1,lineage2,102


In [28]:
Pek18CI_AsmSumm["NumSNPs_PilonPolished"].describe()

count     17.000000
mean     136.470588
std       93.546190
min       50.000000
25%       61.000000
50%      103.000000
75%      214.000000
max      313.000000
Name: NumSNPs_PilonPolished, dtype: float64

In [29]:
IDs_Peker_SuspectAssembly = ["8644-04"]

Pek17CI_AsmSumm = Pek18CI_AsmSumm[ ~Pek18CI_AsmSumm["SampleID"].isin(IDs_Peker_SuspectAssembly)]

Pek17CI_SampleIDs = list( Pek17CI_AsmSumm["SampleID"].values )

Pek17CI_AsmSumm.shape  

(17, 27)

## 1.C) Parse Assembly INFO for Hall80CI dataset (N = 80)

In [30]:

Hall80CI_AsmSumm = pd.read_csv(Hall80CI_AsmSummary_TSV_PATH, sep = "\t")

Hall80CI_AsmSumm = Hall80CI_AsmSumm.sort_values("Lineage_ONTAsm_WiPilonPolish")

Hall80CI_SampleIDs = list( Hall80CI_AsmSumm["SampleID"].values )

Hall80CI_AsmSumm["AsmApproach"] = "ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish"

Hall80CI_AsmSumm["Lineage"] = Hall80CI_AsmSumm["Lineage_ONTAsm_WiPilonPolish"]

Hall80CI_AsmSumm["PrimaryLineage"] = Hall80CI_AsmSumm["PrimaryLineage_ONTAsm_WiPilonPolish"]

print("# of total samples:", len(Hall80CI_SampleIDs) )

#print(', '.join(Hall80CI_SampleIDs) )

# Make sample to lineage mapping dict
Hall80CI_ID_To_PrimLineage_Dict = dict(Hall80CI_AsmSumm[['SampleID', 'PrimaryLineage_ONTAsm_WiPilonPolish']].values)
Hall80CI_ID_To_Lineage_Dict = dict(Hall80CI_AsmSumm[['SampleID', 'Lineage_ONTAsm_WiPilonPolish']].values)
Hall80CI_ID_To_Dataset_Dict = dict(Hall80CI_AsmSumm[['SampleID', 'Dataset_Tag']].values)


Hall80CI_AsmSumm["Mean_SR_Cov"] = Hall80CI_AsmSumm["IlluminaCov_To_ONTAsm"]


# of total samples: 78


In [31]:
Hall80CI_AsmSumm.head(1)

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_ONTAsm,Lineage_ONTAsm_WiPilonPolish,FlyeI3_dnaA_Found,FlyeI3M_dnaA_Found,FlyeI3MPP_dnaA_Found,IlluminaCov_To_ONTAsm,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,NumChanges_PolyPolished,PrimaryLineage_ONTAsm_WiPilonPolish,Dataset_Tag,AsmApproach,Lineage,PrimaryLineage,Mean_SR_Cov
41,mada_1-10,1,4431642,146,150,1734,582,lineage1.1.2,lineage1.1.2,False,True,True,48,1092,10,22,21,1,1060,1010,9,lineage1,Hall2022,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,lineage1.1.2,lineage1,48


## 1.D) Subset the Hall-80CI to the Hall-78CI (Remove suspect Asms)

In [32]:
IDs_Hall_SuspectAssembly = ["mada_1-40", "mada_1-2", "8644-04"]
IDs_Hall_SuspectAssembly = ["mada_1-40", "mada_1-2"] #"8644-04"]

Hall78CI_AsmSumm = Hall80CI_AsmSumm[ ~Hall80CI_AsmSumm["SampleID"].isin(IDs_Hall_SuspectAssembly)]

Hall78CI_SampleIDs = list( Hall78CI_AsmSumm["SampleID"].values )

Hall78CI_AsmSumm.shape  

(78, 27)

In [33]:
# ONT Isolates/Assemblies that were removed: "8644-04", "mada_1-40", "mada_1-2", They all had really clear misassemblies that didn't reflect the actual consensus of the long-reads.


## Create list of SampleIDs for ONT runs

In [34]:
ONT_SampleIDs_95CI = Hall78CI_SampleIDs + Pek17CI_SampleIDs
len(ONT_SampleIDs_95CI)

95

## 1.E) Parse Assembly INFO for `TRUST_CCS_8CI` (TRUST, South Africa) dataset (N = 8)

In [35]:
ZAHF_8CI_AsmSumm_DF = pd.read_csv(TRUST_PBSet1_8CI_AsmSumm_TSV_PATH, sep = "\t")

ZAHF_8CI_SampleIDs = list( ZAHF_8CI_AsmSumm_DF["SampleID"].values )

ZAHF_8CI_AsmSumm_DF["AsmApproach"] = "PBccs_LR_Flye_I3_SR_Pilon"

ZAHF_8CI_AsmSumm_DF["Lineage"] = ZAHF_8CI_AsmSumm_DF["Lineage_AsmPP"]

ZAHF_8CI_AsmSumm_DF["PrimaryLineage"] = ZAHF_8CI_AsmSumm_DF["PrimaryLineage_Asm"]

print("# of total samples:", len(ZAHF_8CI_SampleIDs) )

print(', '.join(ZAHF_8CI_SampleIDs) )

# Make sample to lineage mapping dict
ZAHF_8CI_ID_To_PrimLineage_Dict = dict(ZAHF_8CI_AsmSumm_DF[['SampleID', 'PrimaryLineage_Asm']].values)
ZAHF_8CI_ID_To_Lineage_Dict = dict(ZAHF_8CI_AsmSumm_DF[['SampleID', 'Lineage_Asm']].values)
ZAHF_8CI_ID_To_Dataset_Dict = dict(ZAHF_8CI_AsmSumm_DF[['SampleID', 'Dataset_Tag']].values)

# Filler value!
ZAHF_8CI_AsmSumm_DF["Mean_SR_Cov"] = 250 #ZAHF_8CI_AsmSumm_DF["IlluminaCov_To_ONTAsm"]


# of total samples: 8
S0070-08, S0085-01, S0107-01, S0089-01, S0256-08, S0123-01, S0106-01, S0262-02


In [36]:
ZAHF_8CI_AsmSumm_DF#.head(12)

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,FlyeI3_dnaA_Found,FlyeI3M_dnaA_Found,FlyeI3MPP_dnaA_Found,IlluminaCov_To_ONTAsm,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,PrimaryLineage_Asm,Dataset_Tag,PatientID,IsolateNum,MFS_ID,AsmApproach,Lineage,PrimaryLineage,Mean_SR_Cov
0,S0070-08,1,4424401,69,76,2414,1193,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,1,0,0,0,0,1,1,lineage2,TRUST_PB_Set1,S0070,8,MFS-42,PBccs_LR_Flye_I3_SR_Pilon,lineage2.2.1,lineage2,250
1,S0085-01,1,4417383,39,38,3000,1431,lineage2.2.1.1,lineage2.2.1.1,True,NaN,NaN,NaN,1,0,1,1,0,0,0,lineage2,TRUST_PB_Set1,S0085,1,MFS-51,PBccs_LR_Flye_I3_SR_Pilon,lineage2.2.1.1,lineage2,250
2,S0107-01,1,4416462,64,62,2816,1350,lineage2.2.1.1,lineage2.2.1.1,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage2,TRUST_PB_Set1,S0107,1,MFS-58,PBccs_LR_Flye_I3_SR_Pilon,lineage2.2.1.1,lineage2,250
3,S0089-01,1,4415290,40,39,2741,1346,lineage2.2.1.1,lineage2.2.1.1,True,NaN,NaN,NaN,1,0,0,0,0,1,1,lineage2,TRUST_PB_Set1,S0089,1,MFS-54,PBccs_LR_Flye_I3_SR_Pilon,lineage2.2.1.1,lineage2,250
4,S0256-08,1,4413248,53,51,2916,1452,lineage2.2.1.1,lineage2.2.1.1,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage2,TRUST_PB_Set1,S0256,8,MFS-173,PBccs_LR_Flye_I3_SR_Pilon,lineage2.2.1.1,lineage2,250
5,S0123-01,1,4402640,57,55,3084,1478,lineage4.4.1.1,lineage4.4.1.1,True,NaN,NaN,NaN,1,0,1,1,0,0,0,lineage4,TRUST_PB_Set1,S0123,1,MFS-60,PBccs_LR_Flye_I3_SR_Pilon,lineage4.4.1.1,lineage4,250
6,S0106-01,1,4412324,59,57,3123,1497,lineage4.1.1.3,lineage4.1.1.3,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage4,TRUST_PB_Set1,S0106,1,MFS-56,PBccs_LR_Flye_I3_SR_Pilon,lineage4.1.1.3,lineage4,250
7,S0262-02,1,4402641,44,42,3751,1676,lineage4.4.1.1,lineage4.4.1.1,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage4,TRUST_PB_Set1,S0262,2,MFS-181,PBccs_LR_Flye_I3_SR_Pilon,lineage4.4.1.1,lineage4,250


In [37]:
ZAHF_8CI_AsmSumm_DF.head(2)

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,FlyeI3_dnaA_Found,FlyeI3M_dnaA_Found,FlyeI3MPP_dnaA_Found,IlluminaCov_To_ONTAsm,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,PrimaryLineage_Asm,Dataset_Tag,PatientID,IsolateNum,MFS_ID,AsmApproach,Lineage,PrimaryLineage,Mean_SR_Cov
0,S0070-08,1,4424401,69,76,2414,1193,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,1,0,0,0,0,1,1,lineage2,TRUST_PB_Set1,S0070,8,MFS-42,PBccs_LR_Flye_I3_SR_Pilon,lineage2.2.1,lineage2,250
1,S0085-01,1,4417383,39,38,3000,1431,lineage2.2.1.1,lineage2.2.1.1,True,NaN,NaN,NaN,1,0,1,1,0,0,0,lineage2,TRUST_PB_Set1,S0085,1,MFS-51,PBccs_LR_Flye_I3_SR_Pilon,lineage2.2.1.1,lineage2,250


In [38]:
ZAHF_8CI_IsolateID_To_MFSID_Dict = dict(ZAHF_8CI_AsmSumm_DF[['SampleID', 'MFS_ID']].values)  

In [39]:
ZAHF_8CI_IsolateID_To_MFSID_Dict

{'S0070-08': 'MFS-42',
 'S0085-01': 'MFS-51',
 'S0107-01': 'MFS-58',
 'S0089-01': 'MFS-54',
 'S0256-08': 'MFS-173',
 'S0123-01': 'MFS-60',
 'S0106-01': 'MFS-56',
 'S0262-02': 'MFS-181'}

In [40]:
",".join(ZAHF_8CI_AsmSumm_DF["SampleID"].values)

'S0070-08,S0085-01,S0107-01,S0089-01,S0256-08,S0123-01,S0106-01,S0262-02'

In [41]:
",".join(ZAHF_8CI_AsmSumm_DF["MFS_ID"].values)

'MFS-42,MFS-51,MFS-58,MFS-54,MFS-173,MFS-60,MFS-56,MFS-181'

In [42]:
PMP48CI_AsmSumm.head(4)

,Alt_SampleID,numContigs_Complete,circContig_Length,circContig_Cov,PacBio_Subread_Median_Length,LineageCall_Illumina,LineageCall_PacBio,F2_Illumina,F2_PacBio,ANI_I3,ANI_I3_PP,IlluminaWGSToH37rv_AvrgCov,PacBio_Subreads_H37Rv_AvrgCov,NumAnno_ORFs_PB_PilonPolished,NumAnno_ORFs_PB_DeNovo,GCcontent_PB_PP_GBK,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,PrimaryLineage_PB,PrimaryLineage_Ill,Dataset_Tag,AsmApproach,Lineage,PrimaryLineage,SampleID,Mean_SR_Cov
0,N0072,1,4421406,358,2413.0,"lineage1,lineage1.1,lineage1.1.2","lineage1,lineage1.1,lineage1.1.2",0.021909,0.263559,99.8849,99.8852,112,348,4047,4051,65.610969,11,0,4,3,1,7,7,lineage1,lineage1,ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon,"lineage1,lineage1.1,lineage1.1.2",lineage1,N0072,112
1,N0153,1,4389210,372,2027.0,"lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1","lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1",0.023551,0.272487,99.8687,99.8692,98,370,4037,4043,65.612081,53,19,2,1,1,32,32,lineage1,lineage1,ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon,"lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1",lineage1,N0153,98
15,DNA089,1,4420000,933,9945.0,"lineage2,lineage2.2,lineage2.2.1","lineage2,lineage2.2,lineage2.2.1",0.009493,0.200000,99.8898,99.8920,65,1573,4078,4107,65.609800,79,2,0,0,0,77,76,lineage2,lineage2,TB_Portals_2020,PBrs2_LR_Flye_I3_SR_Pilon,"lineage2,lineage2.2,lineage2.2.1",lineage2,TB3113,65
13,ZRB10578980,1,4413217,374,3790.0,"lineage2,lineage2.2,lineage2.2.1","lineage2,lineage2.2,lineage2.2.1",0.009687,0.216928,99.8957,99.8918,69,363,4066,4106,65.606149,106,4,16,10,2,86,85,lineage2,lineage2,TB_Portals_2020,PBrs2_LR_Flye_I3_SR_Pilon,"lineage2,lineage2.2,lineage2.2.1",lineage2,TB1236,69


In [43]:
PMP48CI_AsmSumm["Dataset_Tag"].value_counts()

Dataset_Tag
TB_Portals_2020         21
Farhat_Peru_2019        13
ChinerOms_2019          12
Lee2020_Elife            1
Ngabonziza_Lin8_2020     1
Name: count, dtype: int64

## Create a merged Sample info DF

In [44]:
#colToMergeOn = ['SampleID', 'numContigs_Complete',
#                'circContig_Length', 'circContig_Cov', 'Dataset_Tag', "AsmApproach"]

colToMergeOn = ['SampleID', 'numContigs_Complete', 'circContig_Cov',
                "PrimaryLineage", "Lineage",
                 'Dataset_Tag', "AsmApproach", 
                "circContig_Cov", "Mean_SR_Cov", ]

listOf_AsmDFs_TrimCol = [PMP48CI_AsmSumm[colToMergeOn],
                         Pek17CI_AsmSumm[colToMergeOn],
                         Hall78CI_AsmSumm[colToMergeOn],
                         ZAHF_8CI_AsmSumm_DF[colToMergeOn]]

All_WGS_and_AsmInfo_DF = pd.concat(listOf_AsmDFs_TrimCol)

All_WGS_and_AsmInfo_DF.columns = ['SampleID', 'numContigs_Complete', 'Flye_CircContig_Cov',
                          "PrimaryLineage", "Lineage",
                          'Dataset_Tag', "AsmApproach",  "circContig_Cov", "Mean_SR_Cov", ]

All_WGS_and_AsmInfo_DF.shape

(151, 9)

In [45]:
All_WGS_and_AsmInfo_DF["Dataset_Tag"].value_counts()

Dataset_Tag
Hall2022                78
TB_Portals_2020         21
Peker2021               17
Farhat_Peru_2019        13
ChinerOms_2019          12
TRUST_PB_Set1            8
Lee2020_Elife            1
Ngabonziza_Lin8_2020     1
Name: count, dtype: int64

In [46]:
All_WGS_and_AsmInfo_DF["AsmApproach"].value_counts()

AsmApproach
ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish    95
PBrs2_LR_Flye_I3_SR_Pilon                48
PBccs_LR_Flye_I3_SR_Pilon                 8
Name: count, dtype: int64

In [47]:
All_WGS_and_AsmInfo_DF.head()

,SampleID,numContigs_Complete,Flye_CircContig_Cov,PrimaryLineage,Lineage,Dataset_Tag,AsmApproach,circContig_Cov,Mean_SR_Cov
0,N0072,1,358,lineage1,"lineage1,lineage1.1,lineage1.1.2",ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon,358,112
1,N0153,1,372,lineage1,"lineage1,lineage1.1,lineage1.1.1,lineage1.1.1.1",ChinerOms_2019,PBrs2_LR_Flye_I3_SR_Pilon,372,98
15,TB3113,1,933,lineage2,"lineage2,lineage2.2,lineage2.2.1",TB_Portals_2020,PBrs2_LR_Flye_I3_SR_Pilon,933,65
13,TB1236,1,374,lineage2,"lineage2,lineage2.2,lineage2.2.1",TB_Portals_2020,PBrs2_LR_Flye_I3_SR_Pilon,374,69
12,TB2659,1,421,lineage2,"lineage2,lineage2.2,lineage2.2.1",TB_Portals_2020,PBrs2_LR_Flye_I3_SR_Pilon,421,69


In [48]:
All_WGS_and_AsmInfo_DF["circContig_Cov"].describe()

count    151.000000
mean     221.867550
std      222.262058
min       25.000000
25%       99.500000
50%      149.000000
75%      230.500000
max      964.000000
Name: circContig_Cov, dtype: float64

In [49]:
All_WGS_and_AsmInfo_DF["Mean_SR_Cov"].describe()

count    151.000000
mean      85.993377
std       73.944889
min       22.000000
25%       47.000000
50%       56.000000
75%       95.500000
max      664.000000
Name: Mean_SR_Cov, dtype: float64

In [50]:
All_WGS_and_AsmInfo_DF.query("circContig_Cov < 40")

,SampleID,numContigs_Complete,Flye_CircContig_Cov,PrimaryLineage,Lineage,Dataset_Tag,AsmApproach,circContig_Cov,Mean_SR_Cov
2,QC-9,1,37,lineage3,lineage3,Peker2021,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,37,106
3,QC-8,1,35,lineage4,lineage4.1.2.1,Peker2021,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,35,96
1,QC-10,1,30,lineage4,lineage4.1.2.1,Peker2021,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,30,86
5,QC-6,1,25,lineage4,lineage4.3.3,Peker2021,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,25,79
11,R21893,1,37,lineage2,lineage2.2.2,Hall2022,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,37,39
74,mada_2-31,1,33,lineage4,lineage4.1,Hall2022,ONT9.4_LR_FlyeI3M_SR_Pilon_PolyPolish,33,45
1,S0085-01,1,39,lineage2,lineage2.2.1.1,TRUST_PB_Set1,PBccs_LR_Flye_I3_SR_Pilon,39,250
